# QickSim AWG Tuning Demo

This notebook shows the hardware-free `QickSim` flow for an AWG tuning bit/HWH pair. Set `BITFILE` to the `.bit` file you want to simulate. `QickSim` will look for the matching `.hwh` file next to it.

Authors: Jeonghyun Park (jeonghyun.park@ubc.ca or alexist@snu.ac.kr), Farbod


In [1]:
from pathlib import Path
import sys

# When running this notebook directly from the repository checkout, make qick_lib importable.
cwd = Path.cwd().resolve()
for candidate in [cwd / "qick_lib", cwd.parent / "qick_lib", cwd.parent / "qick" / "qick_lib"]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from qick.sim import QickSim
from qick.sim.models import TimedCommandEvent

c:\JeonghyunPark\Anaconda\envs\qick_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Change this to your AWG tuning bitstream path.
# The matching .hwh must be in the same directory unless you pass hwhfile=... to QickSim.
BITFILE = Path(r"C:\JeonghyunPark\Workspace\QSTL_QICK\qick\firmware\projects\qstl_awg_tuning\bitstream.bit")

if not BITFILE.exists():
    raise FileNotFoundError(f"Set BITFILE to an existing .bit file: {BITFILE}")

sim = QickSim(BITFILE, strict=False)
print("gens:", len(sim["gens"]))
print("readouts:", len(sim["readouts"]))
print("avg_bufs:", len(sim["avg_bufs"]))
print("awg_tunings:", len(sim["awg_tunings"]))

gens: 12
readouts: 4
avg_bufs: 4
awg_tunings: 8


In [3]:
print(sim)

QICK running on SIM, software version 0.2.357

Firmware configuration (built Wed Jun 24 10:18:26 2026):
	Groups of related clocks: [tProc 0], [DAC tile 0], [ADC tile 0]

	12 signal generator channels:
	0:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 16384 complex samples (2.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 0 is simulated RFDC DAC 00
	1:	axis_awg_tuning_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		DAC tile 1, blk 0 is simulated RFDC DAC 10
	2:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 16384 complex samples (2.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 1 is simulated RFDC DAC 01
	3:	axis_awg_tuning_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		DAC tile 1, blk 1 is simulated RFDC DAC 11
	4:	axis_signal_gen_v6 - fs=6144.000 Msps, fabric=384.000 MHz
		envelope memory: 16384 complex samples (2.667 us)
		32-bit DDS, range=6144.000 MHz
		DAC tile 0, blk 3 is simulated RFDC DAC 03
	5:	axis_

In [7]:
awg_indices = [idx for idx, gen in enumerate(sim["gens"]) if gen.get("gen_type") == "awg_tuning"]
print("AWG tuning gen indices:", awg_indices)

if not awg_indices:
    raise RuntimeError("No AWG tuning channels were discovered in this bitstream/HWH.")

awg_idx = awg_indices[0]
awg = sim.gens[awg_idx]
print("selected AWG gen index:", awg_idx)
print("tproc_ch:", awg["tproc_ch"])
print("tmux_ch:", awg.cfg.get("tmux_ch"))
print("n_pts:", awg.cfg.get("n_pts"))

AWG tuning gen indices: [1, 3, 5, 7, 8, 9, 10, 11]
selected AWG gen index: 1
tproc_ch: 0
tmux_ch: 1
n_pts: 16


## AveragerProgram-style AWG tuning example

This mirrors the usual QICK program pattern: define an `AveragerProgram`, put one-time setup in `initialize()`, put the repeated body in `body()`, then simulate the compiled ASM v1 program with `sim.simulate_program(prog, ...)`.

In [ ]:
from qick import AveragerProgram


class AWGTuning_Loopback_Test(AveragerProgram):
    def initialize(self):
        self.awg_ch = self.cfg["awg_ch"]

        # Initial DC-like SET value. The SET duration is used for scheduling only;
        # the AWG tuning RTL holds this value until the next command.
        self.awg_set(
            ch=self.awg_ch,
            value=self.cfg["start_value"],
            duration=self.cfg["set_duration"],
            t=20,
        )

        self.synci(100)

    def body(self):
        # RAMP starts from the AWG tuning internal current value and moves to target_value.
        self.awg_ramp(
            ch=self.awg_ch,
            target=self.cfg["target_value"],
            duration=self.cfg["ramp_duration"],
            t=100,
        )

        self.sync_all()


prog_cfg = {
    "reps": 1,
    "awg_ch": awg_idx,
    "start_value": 1000,
    "target_value": 4000,
    "set_duration": 64,
    "ramp_duration": 160,
}

prog = AWGTuning_Loopback_Test(sim, prog_cfg)
print("program instruction count:", len(prog.prog_list))

In [ ]:
# Print the compiled ASM v1 program.
print(prog)

In [ ]:
prog_result = sim.simulate_program(prog, cycles=320)
prog_result.summary()

In [ ]:
prog_lanes = prog_result.channel_results["lane_samples"][f"gen{awg_idx}"]
print("first 32 lane-0 samples from AveragerProgram simulation:")
print(prog_lanes[:32, 0].tolist())
print("samples around the ramp start:")
print(prog_lanes[100:112, 0].tolist())

## Low-level event API example

The cells below show the lower-level timed-event API. Most users should prefer the `AveragerProgram` example above.

In [ ]:
# Build two low-level timed command events using the normal AWG tuning driver helpers.
set_cmd = awg.set_cmd(1000)
ramp_cmd = awg.ramp_cmd(4000, 160, step=0)

tmux_ch = awg.cfg.get("tmux_ch")
if tmux_ch is not None:
    set_cmd |= (tmux_ch & 0xFF) << 152
    ramp_cmd |= (tmux_ch & 0xFF) << 152

events = [
    TimedCommandEvent(cycle=10, word=set_cmd, tproc_ch=awg["tproc_ch"], label="awg_set"),
    TimedCommandEvent(cycle=40, word=ramp_cmd, tproc_ch=awg["tproc_ch"], label="awg_ramp"),
]

result = sim.simulate_events(events, cycles=128)
result.summary()

In [9]:
lanes = result.channel_results["lane_samples"].get(f"gen{awg_idx}")
if lanes is None:
    raise RuntimeError("No lane samples were produced for the selected AWG channel.")

print("first 24 lane-0 samples:")
print(lanes[:24, 0].tolist())

first 24 lane-0 samples:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]


In [ ]:
# Optional CSV export. This writes files such as qicksim_awg_tuning_awg_packed.csv.
# paths = result.to_csv(Path("qicksim_awg_tuning"))
# paths